# 01 – Data Exploration & Cleaning (Combined)

This notebook loads the **Online Retail II** dataset, cleans it step‑by‑step (each transformation in its own cell for easy inspection), and then performs the full exploratory analysis with premium Plotly visualisations.

All visualisations use the dark theme defined in `src/visualisations.py`.

In [1]:
import pandas as pd, numpy as np
import sys, os
sys.path.append(os.path.abspath('..'))

import plotly.io as pio
# Use a dark theme for all Plotly figures
pio.templates.default = 'plotly_dark'

# Load the raw dataset (same URL as the original notebook)
DATA_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx'
print('Loading raw dataset…')
df_raw = pd.read_excel(DATA_URL, sheet_name='Year 2010-2011')
print(f'Raw shape: {df_raw.shape}')

Loading raw dataset…
Raw shape: (541910, 8)


## Quick glance at the raw data

In [2]:
df_raw.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


##  Data Cleaning – step‑by‑step
Each block mirrors the logic in `src/data_cleaning.py` but is split into its own cell so you can run and verify it independently.

In [3]:
# Work on a copy of the raw dataframe
df = df_raw.copy()

###  Handle missing **CustomerID**
* Rename `Customer ID` → `CustomerID` (if present).
* Fill missing values with the placeholder **`Guest`**.

In [4]:
if 'Customer ID' in df.columns:
    df.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

if 'CustomerID' in df.columns:
    df['CustomerID'] = df['CustomerID'].fillna('Guest')

print('CustomerID normalised – missing values filled')

CustomerID normalised – missing values filled


###  Normalise price column name
The source file uses `Price`; the rest of the codebase expects `UnitPrice`. Rename if needed.

In [5]:
if 'Price' in df.columns and 'UnitPrice' not in df.columns:
    df.rename(columns={'Price': 'UnitPrice'}, inplace=True)
    print('Renamed Price → UnitPrice')
else:
    print('Price column already named UnitPrice')

Renamed Price → UnitPrice


###  Remove duplicate rows

In [6]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Removed {before - after:,} duplicate rows')

Removed 5,268 duplicate rows


###  Flag negative **Quantity** as returns (add `IsReturn` flag)

In [7]:
if 'Quantity' in df.columns:
    df['IsReturn'] = df['Quantity'] < 0
    return_count = df['IsReturn'].sum()
    print(f'{return_count:,} rows flagged as returns')

10,587 rows flagged as returns


###  Filter out rows where **UnitPrice** ≤ 0

In [8]:
if 'UnitPrice' in df.columns:
    df = df[df['UnitPrice'] > 0].copy()
    print(f'Rows after UnitPrice filter: {len(df):,}')

Rows after UnitPrice filter: 534,130


###  Convert **InvoiceDate** to datetime

In [9]:
if 'InvoiceDate' in df.columns:
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
    print('InvoiceDate converted to datetime')

InvoiceDate converted to datetime


###  Create **Revenue** column (Quantity × UnitPrice)

In [10]:
if 'Quantity' in df.columns and 'UnitPrice' in df.columns:
    df['Revenue'] = df['Quantity'] * df['UnitPrice']
    print('Revenue column added')

Revenue column added


##  Cleaned data preview

In [11]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34


---
##  Exploration – basic overview

In [12]:
print('Shape:', df.shape)
df.info()

Shape: (534130, 10)
<class 'pandas.core.frame.DataFrame'>
Index: 534130 entries, 0 to 541909
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      534130 non-null  object        
 1   StockCode    534130 non-null  object        
 2   Description  534130 non-null  object        
 3   Quantity     534130 non-null  int64         
 4   InvoiceDate  534130 non-null  datetime64[ns]
 5   UnitPrice    534130 non-null  float64       
 6   CustomerID   534130 non-null  object        
 7   Country      534130 non-null  object        
 8   IsReturn     534130 non-null  bool          
 9   Revenue      534130 non-null  float64       
dtypes: bool(1), datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 41.3+ MB


### Missing values per column

In [13]:
df.isna().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
IsReturn       0
Revenue        0
dtype: int64

### Duplicated rows (should be 0 after cleaning)

In [14]:
df.duplicated().sum()

np.int64(0)

### Negative Quantity (returns) count

In [15]:
(df['Quantity'] < 0).sum() if 'Quantity' in df.columns else 'Quantity column missing'

np.int64(9251)

### UnitPrice ≤ 0 count (should be 0 after cleaning)

In [16]:
(df['UnitPrice'] <= 0).sum() if 'UnitPrice' in df.columns else 'UnitPrice column missing'

np.int64(0)

### Save the cleaned dataset for downstream notebooks

In [17]:
import os
os.makedirs('../data', exist_ok=True)
print('Saving cleaned data to ../data/cleaned_retail_data.csv...')
df.to_csv('../data/cleaned_retail_data.csv', index=False)
print('Saved successfully! You can now use this file in downstream notebooks.')

Saving cleaned data to ../data/cleaned_retail_data.csv...
Saved successfully! You can now use this file in downstream notebooks.


---
##  Next steps
* The cleaned `df` can now be fed into the deep EDA notebook (`02_eda_analysis.ipynb`), RFM segmentation notebook, or the Streamlit app.
* Feel free to modify any cleaning step above and see the downstream impact.